<a href="https://colab.research.google.com/github/sara-velmont/Proyectos/blob/main/Red_neuronal_con_la_arquitectura_propuesta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

En este programa se muestra nuevamente el problema de clasificación de tres puntos que se había tratado de resolver con tres neuronas previamente.

In [ ]:
#Paquetería
import numpy as np

Generación de tres clusters de 100 puntos en el espacio cada uno.

In [ ]:
N=100
np.random.rand(N)

#Haciendo 3 clusters de 100 puntos cada uno
x1=2+.2*(2*(np.random.rand(N)-0.5))
y1=1+.2*(2*(np.random.rand(N)-0.5))
z1=-5+.2*(2*(np.random.rand(N)-0.5))


x2=-3+.2*(2*(np.random.rand(N)-0.5))
y2=-4+.2*(2*(np.random.rand(N)-0.5))
z2=2+.2*(2*(np.random.rand(N)-0.5))

x3=-5+.2*(2*(np.random.rand(N)-0.5))
y3=2+.2*(2*(np.random.rand(N)-0.5))
z3=3+.2*(2*(np.random.rand(N)-0.5))

M1=[x1,y1,z1]
A1=np.array(M1)
M2=[x2,y2,z2]
A2=np.array(M2)
M3=[x3,y3,z3]
A3=np.array(M3)

M=np.concatenate([M1,M2,M3],axis=1)

In [ ]:
#Salidas para las 3 distintas clases:
sal1 = np.zeros((3, N)) #Cada una es una matriz 3x100
sal2 = np.zeros((3, N))
sal3 = np.zeros((3, N))
sal1[0, :] = 1 #Matrices de 0's con con una fila de 1's, filas distintas para cada clase
sal2[1, :] = 1
sal3[2, :] = 1

In [ ]:
salidas = np.concatenate([sal1, sal2, sal3], axis=1) #Salidas es una arreglo de 3x300

In [ ]:
from sklearn.utils import shuffle #Revolvemos en el mismo orden las columnas de la matriz salidas y la matriz entradas
numeros = np.array(range(3* N))
numeros2 = np.array(range(3* N))
numeros, numeros2 = shuffle(numeros, numeros2, random_state=0)

In [ ]:
#Entradas y salidas
Entradas=M[:,numeros]
Salidas=salidas[:,numeros2] #Matriz 3x300

In [ ]:
Entradas=np.concatenate([Entradas,np.ones((1,300))]) #Se añade el bías a las entradas


In [ ]:
#Definiendo las funciones de activación y la función de costo:
def sigmoide(x):
    return 1 / (1 + np.exp(-x))

def softmax(x):
    exps = np.exp(x - np.max(x, axis=0))
    return exps / np.sum(exps, axis=0) #Realiza la suma sobre las columnas

def entropia_cruzada(d, y_pred):
    epsilon = 1e-10 #Se escoge este valor para evitar problemas de división por cero o logaritmos de valores cercanos a cero.
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon) #np.clip se utiliza para asegurar que los valores de y_pred estén dentro del rango [epsilon, 1 - epsilon]
    return -np.sum(d * np.log(y_pred)) / d.shape[1] #Se utiliza L = -1/300 * Σ[Σ(d * log(y_pred))]


In [ ]:
salidas.shape[1]

300

In [ ]:
#Definición de la configuración de la red

entradas = 3 #características
capa_oculta = 4 #neuronas
capa_salida = 3 #neuronas

In [ ]:
#Creación de los pesos para cada capa

w1 = np.random.rand(capa_oculta, entradas+1) #Tamaño de la capa oculta x entradas+bias (4x4)
w2 = np.random.rand(capa_salida, capa_oculta) #Tamaño de la capa de salida x entradas+bias (3x4)

In [ ]:
w1 #pesos iniciales

array([[0.99501631, 0.24291865, 0.95593678, 0.00858318],
       [0.29818335, 0.39266838, 0.21431137, 0.48574979],
       [0.90111719, 0.81044034, 0.4757265 , 0.56558258],
       [0.84840602, 0.62322245, 0.50822947, 0.42082914]])

In [ ]:
w2 #pesos iniciales

array([[0.59819228, 0.68883166, 0.67579477, 0.65771996],
       [0.32987465, 0.90298624, 0.50142582, 0.1278965 ],
       [0.85272609, 0.22696315, 0.50984222, 0.52828188]])

In [ ]:
Entradas.shape

(4, 300)

In [ ]:
#Forward propagation

z1 = np.dot(w1, Entradas) #Multiplicación de entradas por los pesos en la primera capa (4x4)x(4x300)
y1 = sigmoide(z1) #Salida de la capa oculta
z2 = np.dot(w2, y1) #Multiplicación de los pesos de la segunda capa por la salida de la primera capa (3x4)x(4x300)
y2 = softmax(z2) #Salida de la red
Coste = entropia_cruzada(Salidas,y2) #La función de costo con los parámetros determinados


In [ ]:
#Algortimo Backpropagation, derivadas parciales de la función de Costo

dz2 = y2 - Salidas #Error
dw2 = (1/N) * np.dot(dz2, y1.T) #Derivada parcial respecto de los pesos de la capa de salida 1/N normalización de la derivada
dy1 = np.dot(w2.T, dz2) #Derivada parcial respecto a la función de activación sigmoide
dz1 = dy1 * sigmoide(z1) * (1 - sigmoide(z1)) #Derivada parcial respecto suma ponderada capa oculta
dw1 = (1/N) * np.dot(dz1, Entradas.T) #Derivada parcial respecto a los pesos de la primera capa

In [ ]:
#Actualización de los pesos con el descenso del gradiente de la función de costo ya calculado
r=0.1
w1 -= r*dw1
w2 -= r*dw2

In [ ]:
# Prueba de la red neuronal
epocas=0
historial_pec=[]
tol=0.003

while True:
  z1 = np.dot(w1, Entradas)
  y1 = sigmoide(z1)
  z2 = np.dot(w2, y1)
  y2 = softmax(z2)
  dz2 = y2 - Salidas
  dw2 = (1/N) * np.dot(dz2, y1.T)
  dy1 = np.dot(w2.T, dz2)
  dz1 = dy1 * sigmoide(z1) * (1 - sigmoide(z1))
  dw1 = (1/N) * np.dot(dz1, Entradas.T)
  Coste = entropia_cruzada(Salidas,y2)
  w1 -= r*dw1
  w2 -= r*dw2
  historial_pec.append(Coste)
  if Coste < tol: #Criterio de paro
    break

  print(f"Epoca: {epocas+1}, Valor de la función de costo: {Coste}")
  epocas +=1


Epoca: 1, Valor de la función de costo: 0.1572157096620166
Epoca: 2, Valor de la función de costo: 0.15333216287691462
Epoca: 3, Valor de la función de costo: 0.1496280450909738
Epoca: 4, Valor de la función de costo: 0.1460911956895309
Epoca: 5, Valor de la función de costo: 0.1427104929473496
Epoca: 6, Valor de la función de costo: 0.13947576256935093
Epoca: 7, Valor de la función de costo: 0.13637769146054032
Epoca: 8, Valor de la función de costo: 0.1334077474453031
Epoca: 9, Valor de la función de costo: 0.13055810524355158
Epoca: 10, Valor de la función de costo: 0.12782157872982236
Epoca: 11, Valor de la función de costo: 0.12519155931328924
Epoca: 12, Valor de la función de costo: 0.12266196015491654
Epoca: 13, Valor de la función de costo: 0.12022716586355175
Epoca: 14, Valor de la función de costo: 0.11788198727203023
Epoca: 15, Valor de la función de costo: 0.1156216208775398
Epoca: 16, Valor de la función de costo: 0.11344161253045738
Epoca: 17, Valor de la función de costo

In [ ]:
# Clasificación de las muestras
salidas_red = np.argmax(y2, axis=0) #Muestra el argumento máximo de las columnas o la entrada de la columna más alta
salidas_reales = np.argmax(Salidas, axis=0) #Lo mismo para la matriz de salidas

# Calcular la precisión
Accuracy = np.mean(salidas_red == salidas_reales) #Compara si los argumentos más altos para cada columna de ambas matrices son los mismos
print(f"Prueba de exactitud: {Accuracy}")

Prueba de exactitud: 1.0


Se muestra que con esta arquitectura la red neuronal en esta ocasión logró clasificar correctamente todos los puntos de prueba.